# Pre-TEST3 — Qwen forced-fallback cache (80 Tables)

**Runtime:** Colab GPU (T4) + Linux.

This notebook runs the high-value experiment:
1. `generate_cache()` — Stage3 prompts, rules never consulted
2. `replay_cache()` — same codes on Stage3 / TEST1 / TEST2 executors
3. Download `generations_*.json` + replay CSV for local H1–H6 analysis

Not a private submission. Not the 40s solve benchmark.

In [4]:
!git clone https://github.com/BeediGoua/TEST-PROJET-ML.git


fatal: destination path 'TEST-PROJET-ML' already exists and is not an empty directory.


In [5]:
from google.colab import files
import shutil, os

# Ouvre le notebook dans Colab correctement :
os.chdir('/content')
print('OK — maintenant utilise Fichier → Ouvrir')

OK — maintenant utilise Fichier → Ouvrir


## 0. Mount / upload project

Upload a zip of `projects/` (must include `audit_independent/`, `CODEX_AUDIT_HANDOFF/`, `src/`) or clone your private mirror.
Set `PROJECT_ROOT` below.

In [6]:
import os, sys, platform, subprocess
from pathlib import Path

# --- EDIT THIS ---
# Clone: !git clone https://github.com/BeediGoua/TEST-PROJET-ML.git
# Ce repo = contenu de `projects/` → PROJECT_ROOT = racine du clone
PROJECT_ROOT = Path('/content/TEST-PROJET-ML')
# -----------------

assert PROJECT_ROOT.exists(), f'missing {PROJECT_ROOT}'
AUDIT = PROJECT_ROOT / 'audit_independent'
sys.path.insert(0, str(AUDIT))
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(AUDIT)
print('cwd', Path.cwd())
print('platform', platform.platform())

import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'Enable GPU runtime (Runtime → Change runtime type → T4)'
assert platform.system() == 'Linux'

cwd /content/TEST-PROJET-ML/audit_independent
platform Linux-6.6.122+-x86_64-with-glibc2.39
cuda True Tesla T4


In [7]:
# Dependencies (match local audit / official harness)
%pip -q install 'torch' 'transformers>=4.40' 'accelerate' 'pandas' 'numpy' 'scipy' 'scikit-learn' 'sympy' 'mlarena'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
from gpu_experiments import generate_cache, replay_cache, summarize_replay

cache_path = generate_cache(
    model='Qwen/Qwen2.5-Coder-1.5B-Instruct',
    prompt_agent='stage3',
    seed=0,
)
print('CACHE', cache_path)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[init] modèle Qwen/Qwen2.5-Coder-1.5B-Instruct sur cuda:0, échantillons 4, budget 29.0s
CACHE /content/TEST-PROJET-ML/audit_independent/results/generations_34d6ece55b3c092a.json


In [9]:
rows = replay_cache(cache_path, timeout=15.0)
summary = summarize_replay(rows)
display(summary)
summary.to_csv(str(cache_path).replace('.json', '_summary.csv'))
print('replay rows', len(rows))

,tasks,execution_rate,greedy_accuracy,oracle_samples_at4,oracle_greedy_plus4,vote_accuracy
executor,,,,,,
e4_importlib,80,0.555,0.500,0.7375,0.8250,0.7875
minimal_many,80,0.080,0.025,0.1625,0.1625,0.1625
stage3,80,0.555,0.500,0.7375,0.8250,0.7875
test1_alias_many,80,0.555,0.500,0.7375,0.8250,0.7875
test1_many,80,0.030,0.025,0.0500,0.0500,0.0500
test1_single,80,0.555,0.500,0.7375,0.8250,0.7875
test2_batch,80,0.080,0.025,0.1625,0.1625,0.1625
test2_batch_alias,80,0.080,0.025,0.1625,0.1625,0.1625


replay rows 3200


In [10]:
from google.colab import files
from pathlib import Path
p = Path(cache_path)
for f in [p, Path(str(p).replace('.json', '_replay.csv')), Path(str(p).replace('.json', '_summary.csv'))]:
    if f.exists():
        print('download', f)
        files.download(str(f))
    else:
        print('missing', f)
print('Place generations_*.json under projects/audit_independent/results/ then run diagnostic/07–12 locally.')

download /content/TEST-PROJET-ML/audit_independent/results/generations_34d6ece55b3c092a.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

download /content/TEST-PROJET-ML/audit_independent/results/generations_34d6ece55b3c092a_replay.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

download /content/TEST-PROJET-ML/audit_independent/results/generations_34d6ece55b3c092a_summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Place generations_*.json under projects/audit_independent/results/ then run diagnostic/07–12 locally.
